In [28]:
import pandas as pd
from datetime import timedelta

# ---- CONFIG: map your CSV columns to logical fields ----
COLS = {
    "bike_id": "Bike ID",                      # <- change if your file names it differently (e.g., 'bikeid' or 'Bike Number')
    "start_time": "Start Time",
    "stop_time": "Stop Time",
    "start_station_id": "Start Station ID",
    "end_station_id": "End Station ID",
}
END_TARGETS = {7, 8, 9}
WINDOW = timedelta(hours=1)

# ---- LOAD DATA ----
df = pd.read_csv("/Users/checco/Desktop/project/Scalable_Sys-Course-Project-1/2017-citibike-tripdata/1_January/d.csv")
# Parse times
df[COLS["start_time"]] = pd.to_datetime(df[COLS["start_time"]], utc=True, errors="coerce")
df[COLS["stop_time"]]  = pd.to_datetime(df[COLS["stop_time"]],  utc=True, errors="coerce")

# Coerce station IDs to ints (drop rows we can't parse)
for k in ("start_station_id", "end_station_id"):
    c = COLS[k]
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
df = df.dropna(subset=[COLS["bike_id"], COLS["start_time"], COLS["stop_time"], COLS["start_station_id"], COLS["end_station_id"]])

# Keep only relevant columns
df = df[[COLS["bike_id"], COLS["start_time"], COLS["stop_time"], COLS["start_station_id"], COLS["end_station_id"]]].rename(
    columns={
        COLS["bike_id"]: "bike",
        COLS["start_time"]: "start_time",
        COLS["stop_time"]: "stop_time",
        COLS["start_station_id"]: "start_station",
        COLS["end_station_id"]: "end_station",
    }
)

# ---- CORE: find hotpaths per your pattern ----
# PATTERN: SEQ(BikeTrip+ a[], BikeTrip b)
# WHERE (same bike), adjacency a[i+1].start = a[i].end,
#       a[last].bike = b.bike,
#       b.end in {7,8,9},
#       WITHIN 1h (from a[1].start to b.stop_time)
# RETURN (a[1].start, a[i].end, b.end)

def count_hotpaths_for_bike(bike_df, end_targets=END_TARGETS, window=WINDOW):
    # Sort by start_time to build sequences in temporal order
    bike_df = bike_df.sort_values(["start_time", "stop_time"], kind="mergesort").reset_index(drop=True)

    # We’ll scan with a sliding window that enforces adjacency (end station == next start station).
    # For each start index s, expand as long as adjacency holds; any prefix of length >= 1
    # can be followed by a “b” candidate (current trip j) if end_station in targets
    # AND overall window test passes.
    n = len(bike_df)
    matches = 0

    # Optional: collect details to inspect later
    details = []  # (bike, a1_start_time, a_last_end_station, b_end_station, b_stop_time, chain_len)

    # Pre-extract to speed up Python loops
    start_times = bike_df["start_time"].to_list()
    stop_times  = bike_df["stop_time"].to_list()
    start_stas  = bike_df["start_station"].to_list()
    end_stas    = bike_df["end_station"].to_list()

    for s in range(n):
        # Chain must have at least one a (so j must be >= s)
        # We track adjacency as we move j forward
        chain_len = 1
        a1_start = start_times[s]
        prev_end = end_stas[s]

        # First, consider j=s as potential b ending the chain of length 1
        j = s
        if (int(end_stas[j]) % 10 in end_targets) and ((stop_times[j] - a1_start) <= window):
            matches += 1
            details.append((bike_df.loc[j, "bike"], a1_start, prev_end, end_stas[j], stop_times[j], chain_len))

        # Try to extend the chain: enforce a[i+1].start == a[i].end
        for j in range(s+1, n):
            if start_stas[j] != prev_end:
                break  # adjacency broken; move s forward
            chain_len += 1
            prev_end = end_stas[j]

            # Now j is the candidate “b”
            if (end_stas[j] in end_targets) and ((stop_times[j] - a1_start) <= window):
                matches += 1
                details.append((bike_df.loc[j, "bike"], a1_start, end_stas[j-1], end_stas[j], stop_times[j], chain_len))

    return matches, details

# Apply per bike
counts = []
all_details = []
for bike, g in df.groupby("bike"):
    m, det = count_hotpaths_for_bike(g)
    counts.append({"bike": bike, "matches": m})
    # Keep a small sample of details for sanity checks (optional)
    all_details.extend(det[:50])  # cap to avoid massive objects

counts_df = pd.DataFrame(counts).sort_values("matches", ascending=False).reset_index(drop=True)

print(counts_df.head(20))      # top 20 bikes by matches
# Optionally inspect a few detailed examples
details_df = pd.DataFrame(all_details, columns=["bike","a1_start_time","a_last_end_station","b_end_station","b_stop_time","chain_len"])
print(details_df.head(10))
# Save if needed
# counts_df.to_csv("hotpath_match_counts_by_bike.csv", index=False)
# details_df.to_csv("hotpath_example_details.csv", index=False)


     bike  matches
0   24079       99
1   27018       94
2   26573       93
3   26476       93
4   25482       92
5   25574       91
6   26697       91
7   26041       90
8   26604       89
9   26360       89
10  25882       89
11  26386       89
12  25408       89
13  21983       89
14  27296       88
15  25020       87
16  23650       86
17  27288       85
18  26457       85
19  26603       85
    bike             a1_start_time  a_last_end_station  b_end_station  \
0  14529 2017-01-01 13:43:49+00:00                 319            319   
1  14529 2017-01-02 13:44:50+00:00                 458            458   
2  14529 2017-01-05 06:37:03+00:00                 519            519   
3  14529 2017-01-06 12:09:34+00:00                 297            297   
4  14529 2017-01-09 15:16:55+00:00                 477            477   
5  14529 2017-01-11 18:56:04+00:00                 488            488   
6  14529 2017-01-12 06:37:49+00:00                 458            458   
7  14529 2017-01-

In [30]:
# HOTPATH COUNTER (corrected semantics, no single-trip matches)
# - Reads a CSV of trips
# - Outputs counts per bike and an examples file for spot-checking

import pandas as pd
from datetime import timedelta
from pathlib import Path

# ========= CONFIG =========
CSV_PATH = "d.csv"          # <--- set your file path
SAVE_COUNTS_CSV = "hotpath_match_counts_by_bike.csv"
SAVE_DETAILS_CSV = "hotpath_example_details.csv"   # small sample for auditing
SAMPLE_DETAILS_PER_BIKE = 50     # how many example rows per bike to keep (0 to disable)
WINDOW = timedelta(hours=1)
HOT_LAST_DIGITS = {7, 8, 9}

# If your column names differ, update these. The script will also try to auto-detect 'bike' if possible.
COLS = {
    "bike_id": None,                       # e.g. "Bike ID" or "bikeid" — auto-detected if None
    "start_time": "Start Time",
    "stop_time": "Stop Time",
    "start_station_id": "Start Station ID",
    "end_station_id": "End Station ID",
}

# ========= HELPERS =========

def autodetect_bike_column(df):
    candidates = [
        "Bike ID", "bike_id", "bikeid", "BikeID", "bike", "Bike", "Bike Number", "Bike Number ID"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    # fallback: try any column containing "bike" (case-insensitive)
    for c in df.columns:
        if "bike" in c.lower():
            return c
    raise KeyError(
        "Could not find a bike-id column. Please set COLS['bike_id'] to the correct column name."
    )

def ends_with_789(x):
    """True if station ID ends with 7, 8, or 9 (robust to strings/ints)."""
    try:
        return (int(x) % 10) in HOT_LAST_DIGITS
    except Exception:
        xs = str(x)
        return xs.endswith(("7","8","9"))

def normalize_trips(df):
    # Column mapping (with auto-detect for bike_id)
    bike_col = COLS["bike_id"] or autodetect_bike_column(df)

    # Parse times
    df[COLS["start_time"]] = pd.to_datetime(df[COLS["start_time"]], utc=True, errors="coerce")
    df[COLS["stop_time"]]  = pd.to_datetime(df[COLS["stop_time"]],  utc=True, errors="coerce")

    # Coerce station IDs to numeric (nullable), then to Python ints when used
    for k in ("start_station_id", "end_station_id"):
        c = COLS[k]
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    # Drop rows missing essentials
    df = df.dropna(subset=[bike_col, COLS["start_time"], COLS["stop_time"], COLS["start_station_id"], COLS["end_station_id"]])

    # Keep and rename to canonical columns
    out = df[[bike_col, COLS["start_time"], COLS["stop_time"], COLS["start_station_id"], COLS["end_station_id"]]].rename(
        columns={
            bike_col: "bike",
            COLS["start_time"]: "start_time",
            COLS["stop_time"]: "stop_time",
            COLS["start_station_id"]: "start_station",
            COLS["end_station_id"]: "end_station",
        }
    ).copy()

    # Cast bike to string or int? Use int if numeric, else leave as string
    try:
        out["bike"] = pd.to_numeric(out["bike"], errors="raise").astype("int64")
    except Exception:
        out["bike"] = out["bike"].astype("string")

    # Convert station cols to plain ints for speed (safe because we dropped NaNs)
    out["start_station"] = out["start_station"].astype("int64")
    out["end_station"]   = out["end_station"].astype("int64")

    return out

def count_hotpaths_for_bike(bike_df, window=WINDOW):
    """
    Corrected semantics:
      - require chain length >= 2 (i.e., j starts from s+1)
      - adjacency: a[i+1].start_station == a[i].end_station
      - ends-with 7/8/9 check on b.end_station
      - window: (b.stop_time - a1.start_time) <= 1h
      - count all valid prefixes (overlaps allowed)
    """
    g = bike_df.sort_values(["start_time", "stop_time"], kind="mergesort").reset_index(drop=True)

    n = len(g)
    matches = 0
    details = []  # (bike, a1_start_time, a_last_end_station, b_end_station, b_stop_time, chain_len)

    start_times = g["start_time"].to_list()
    stop_times  = g["stop_time"].to_list()
    start_stas  = g["start_station"].to_list()
    end_stas    = g["end_station"].to_list()
    bike_vals   = g["bike"].to_list()

    for s in range(n):
        a1_start = start_times[s]
        prev_end = end_stas[s]
        chain_len = 1

        # j must be >= s+1 to enforce at least one 'a' before 'b'
        for j in range(s+1, n):
            # enforce adjacency
            if start_stas[j] != prev_end:
                break
            chain_len += 1
            prev_end = end_stas[j]

            # candidate b = trip j (chain_len >= 2 by construction here)
            if ends_with_789(end_stas[j]) and (stop_times[j] - a1_start) <= window:
                matches += 1
                details.append((
                    bike_vals[j],
                    a1_start,
                    end_stas[j-1],     # a[last].end
                    end_stas[j],       # b.end
                    stop_times[j],     # b.stop
                    chain_len
                ))

    return matches, details

# ========= RUN =========

# Load CSV
raw_df = pd.read_csv(CSV_PATH)
df = normalize_trips(raw_df)

# Per-bike computation
counts = []
all_details = []

for bike, g in df.groupby("bike", sort=False):
    m, det = count_hotpaths_for_bike(g)
    counts.append({"bike": bike, "matches": m})

    if SAMPLE_DETAILS_PER_BIKE > 0 and det:
        # keep a small per-bike sample of example details for auditing
        all_details.extend(det[:SAMPLE_DETAILS_PER_BIKE])

counts_df = pd.DataFrame(counts).sort_values("matches", ascending=False).reset_index(drop=True)
display(counts_df.head(30))

print(f"\nTotal bikes with ≥1 match: {(counts_df['matches']>0).sum()} / {len(counts_df)}")
print(f"Total matches (all bikes): {int(counts_df['matches'].sum())}")

# Save outputs
Path(SAVE_COUNTS_CSV).parent.mkdir(parents=True, exist_ok=True)
counts_df.to_csv(SAVE_COUNTS_CSV, index=False)
print(f"\nSaved counts per bike -> {SAVE_COUNTS_CSV}")

if all_details:
    details_df = pd.DataFrame(all_details, columns=["bike","a1_start_time","a_last_end_station","b_end_station","b_stop_time","chain_len"])
    details_df.to_csv(SAVE_DETAILS_CSV, index=False)
    print(f"Saved sample of match details -> {SAVE_DETAILS_CSV}")


,bike,matches
0,19728,1
1,25542,0
2,21136,0
3,18147,0
4,21211,0
5,26819,0
6,16050,0
7,27294,0
8,26501,0
9,23288,0



Total bikes with ≥1 match: 1 / 14
Total matches (all bikes): 1

Saved counts per bike -> hotpath_match_counts_by_bike.csv
Saved sample of match details -> hotpath_example_details.csv


In [21]:
from datetime import timedelta

END_LAST_DIGITS = {7, 8, 9}
WINDOW = timedelta(hours=1)

def ends_with_789(x):
    # robust to strings/ints
    try:
        return (int(x) % 10) in END_LAST_DIGITS
    except Exception:
        xs = str(x)
        return xs.endswith(("7","8","9"))

def count_hotpaths_for_bike(bike_df, window=WINDOW):
    g = bike_df.sort_values(["start_time", "stop_time"], kind="mergesort").reset_index(drop=True)

    n = len(g)
    matches = 0
    details = []  # (bike, a1_start_time, a_last_end_station, b_end_station, b_stop_time, chain_len)

    start_times = g["start_time"].to_list()
    stop_times  = g["stop_time"].to_list()
    start_stas  = g["start_station"].to_list()
    end_stas    = g["end_station"].to_list()
    bike_vals   = g["bike"].to_list()

    for s in range(n):
        a1_start = start_times[s]
        prev_end = end_stas[s]
        chain_len = 1

        # extend the chain forward; b must be a *later* trip (j >= s+1)
        for j in range(s+1, n):
            # enforce adjacency
            if start_stas[j] != prev_end:
                break
            chain_len += 1
            prev_end = end_stas[j]

            # candidate b = trip j (chain_len >= 2 guaranteed here)
            if (ends_with_789(end_stas[j]) and
                (stop_times[j] - a1_start) <= window):
                matches += 1
                details.append((
                    bike_vals[j],
                    a1_start,
                    end_stas[j-1],     # a[last].end
                    end_stas[j],       # b.end
                    stop_times[j],     # b.stop
                    chain_len
                ))

    return matches, details


In [17]:
import re
import pandas as pd

# --- Parse Redis matches.txt ---
pattern = re.compile(r"bike=(\d+)\s+end=(\d+)\s+lenA=(\d+)")
rows = []
with open("../../matches/matches.txt") as f:
    for line in f:
        m = pattern.search(line)
        if m:
            bike, end, lenA = map(int, m.groups())
            rows.append({"bike": bike, "end_station": end, "lenA": lenA})

redis_df = pd.DataFrame(rows)
redis_counts = redis_df.groupby("bike").size().reset_index(name="redis_matches")

# --- Load Pandas result you printed earlier ---
# (If you have it saved as CSV, load that; else paste your top rows)
pandas_counts = pd.DataFrame({
    "bike": [24079,27018,26573,26476,25482,25574,26697,26041,26604,26360,
             25882,26386,25408,21983,27296,25020,23650,27288,26457,26603],
    "pandas_matches": [99,94,93,93,92,91,91,90,89,89,89,89,89,89,88,87,86,85,85,85]
})

# --- Compare ---
merged = pd.merge(pandas_counts, redis_counts, on="bike", how="outer").fillna(0)
merged["delta"] = merged["pandas_matches"] - merged["redis_matches"]

print(merged.sort_values("pandas_matches", ascending=False).head(20))
print("Correlation:", merged["pandas_matches"].corr(merged["redis_matches"]))


       bike  pandas_matches  redis_matches  delta
5587  24079            99.0             45   54.0
7568  27018            94.0             36   58.0
7155  26573            93.0             41   52.0
7063  26476            93.0             37   56.0
6272  25482            92.0             34   58.0
7259  26697            91.0             37   54.0
6357  25574            91.0             32   59.0
6804  26041            90.0             37   53.0
6953  26360            89.0             40   49.0
6977  26386            89.0             32   57.0
6655  25882            89.0             31   58.0
5347  21983            89.0             22   67.0
7183  26604            89.0             42   47.0
6205  25408            89.0             41   48.0
7835  27296            88.0             43   45.0
5848  25020            87.0             27   60.0
5528  23650            86.0             32   54.0
7182  26603            85.0             32   53.0
7045  26457            85.0             30   55.0
